UPscaledEV Project Code for LMP downloading
Under the funding from TotalEnergies
Author: Yizhan Gu
Email: yig031@ucsd.edu
Affiliation: UCSD CER

Readme:
This code aims at downloading LMP data from CAISO OASIS, and unzipping all files. Note that not all files are zippable, that some of them are broken from the website.

Labels:
NOTE: means there's a note and please read it
FIXME: means it's a bug or a problem that needs solving
TODO: means it's a to-do task, but not critical to the code running
VERSION: means there're more than one version for the diversity purpose, possibly shows in objective function choices or results analyses

Acknowledgement:
I gratefully acknowledge the support from my PI Jan Kleissl and TotalEnergies team, and the contributions from Yi-An Chen, whose prior work laid the foundation for this project. Special thanks to the UCSD Grid Lab team members.

In [ ]:

import numpy as np
import pandas as pd
import calendar
from datetime import datetime
from datetime import timedelta
import requests
from tqdm import tqdm
import time
import zipfile
import os

# change the year according to your needs
year = 2025
dir_Input = os.path.join("/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024/2025Data/LMP", str(year))
if not os.path.exists(dir_Input):
    os.makedirs(dir_Input)

os.chdir(dir_Input) # change directory from working dir to dir with files


In [ ]:
# Download data day by day
for month in tqdm(np.array(range(7))+1): # 1:12
    num_days = calendar.monthrange(year,month)[1] 

    for day in np.array(range(num_days))+1: # 1:num_days     
        ThisDate1 = datetime(year,month,day)
        ThisDate2 = ThisDate1 + timedelta(hours=24)
        Y1 = ThisDate1.strftime("%Y")
        M1 = ThisDate1.strftime("%m")
        D1 = ThisDate1.strftime("%d")
        Y2 = ThisDate2.strftime("%Y")
        M2 = ThisDate2.strftime("%m")
        D2 = ThisDate2.strftime("%d")
        
        url_DA = (
    f"http://oasis.caiso.com/oasisapi/SingleZip?resultformat=6"
    f"&queryname=PRC_LMP&version=12"
    f"&startdatetime={Y1}{M1}{D1}T07:00-0000"
    f"&enddatetime={Y2}{M2}{D2}T07:00-0000"
    f"&market_run_id=DAM&node=UCM_6_N001"
)
        url_RT = (
    f"http://oasis.caiso.com/oasisapi/SingleZip?resultformat=6"
    f"&queryname=PRC_RTPD_LMP&version=3"
    f"&startdatetime={Y1}{M1}{D1}T07:00-0000"
    f"&enddatetime={Y2}{M2}{D2}T07:00-0000"
    f"&market_run_id=RTPD&node=UCM_6_N001"
)
        url_FM = (
    f"http://oasis.caiso.com/oasisapi/SingleZip?resultformat=6"
    f"&queryname=PRC_INTVL_LMP&version=3"
    f"&startdatetime={Y1}{M1}{D1}T07:00-0000"
    f"&enddatetime={Y2}{M2}{D2}T07:00-0000"
    f"&market_run_id=RTM&node=UCM_6_N001"
)
        # Create separate folders for DA, RT, FM if they don't exist
        dir_DA = os.path.join(dir_Input, "DA")
        dir_RT = os.path.join(dir_Input, "RT")
        dir_FM = os.path.join(dir_Input, "FM")
        for d in [dir_DA, dir_RT, dir_FM]:
            if not os.path.exists(d):
                os.makedirs(d)

        output_DA = os.path.join(dir_DA, f'LMP_DA_{Y1}{M1}{D1}.zip')
        output_RT = os.path.join(dir_RT, f'LMP_RT_{Y1}{M1}{D1}.zip')
        output_FM = os.path.join(dir_FM, f'LMP_FM_{Y1}{M1}{D1}.zip')
        
        r_DA = requests.get(url_DA)
        r_RT = requests.get(url_RT)
        r_FM = requests.get(url_FM)
        with open(output_DA, 'wb') as f:
            f.write(r_DA.content)
        with open(output_RT, 'wb') as f:
            f.write(r_RT.content)
        with open(output_FM, 'wb') as f:
            f.write(r_FM.content)
            
        time.sleep(5) # can only download 31 days data once, so sleep to avoid server error
            
    print(f"Download month {month} data successful!\n")



  8%|▊         | 1/12 [04:24<48:26, 264.22s/it]

Download month 1 data successful!



 17%|█▋        | 2/12 [08:53<44:32, 267.22s/it]

Download month 2 data successful!



 25%|██▌       | 3/12 [12:51<38:02, 253.63s/it]

...

Download month 10 data successful!



 92%|█████████▏| 11/12 [37:19<03:02, 182.64s/it]

Download month 11 data successful!



100%|██████████| 12/12 [40:20<00:00, 201.74s/it]

Download month 12 data successful!



In [ ]:

# Zip to Csv
extension = ".zip"

# check the ratio of files that are successfully unzipped / all zip files
def csv_to_zip_ratio(directory):
    num_zip = len([f for f in os.listdir(directory) if f.endswith(extension)])
    num_csv = len([f for f in os.listdir(directory) if f.endswith('.csv')])
    if num_zip > 0:
        ratio = num_csv / num_zip * 100
        print(f"Unzip ratio in {directory}: {ratio:.2f}% ({num_csv}/{num_zip})")
    else:
        print(f"No zip files found in {directory}.")
    

for dir_zip in [dir_DA, dir_RT, dir_FM]: # loop through directories
    start_time = time.time()
    with tqdm(total=len([f for f in os.listdir(dir_zip) if f.endswith(extension)]), desc=f"Unzipping in {os.path.basename(dir_zip)}") as pbar:
        pbar.set_postfix({"Time elapsed": "0s"})
        for item in os.listdir(dir_zip): # loop through items in dir
            if item.endswith(extension): # check for ".zip" extension
                file_path = os.path.join(dir_zip, item) # get full path of file
                if zipfile.is_zipfile(file_path): # verify it's actually a zip file
                    try:
                        with zipfile.ZipFile(file_path) as zip_ref: # using context manager
                            zip_ref.extractall(dir_zip) # extract file to dir
                        # print(f"Successfully extracted {item}")
                    except zipfile.BadZipFile:
                        print(f"Error: {item} is not a valid zip file")
                pbar.update(1)
        csv_to_zip_ratio(dir_zip) # check the ratio of files that are successfully unzipped / all zip files
    end_time = time.time()
    elapsed_time = end_time - start_time
    pbar.set_postfix({"Time elapsed": f"{elapsed_time:.2f}s"}) 



Unzipping in DA: 100%|██████████| 365/365 [00:00<00:00, 1918.78it/s, Time elapsed=0s]


Unzip ratio in /Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024/2025Data/LMP/2025/DA: 215.89% (788/365)


Unzipping in RT: 100%|██████████| 365/365 [00:00<00:00, 2701.31it/s, Time elapsed=0s]


Unzip ratio in /Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024/2025Data/LMP/2025/RT: 61.10% (223/365)


Unzipping in FM: 100%|██████████| 365/365 [00:00<00:00, 1576.20it/s, Time elapsed=0s]

Unzip ratio in /Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024/2025Data/LMP/2025/FM: 61.10% (223/365)
